# FactPy SDK Example (Updated)

This example notebook aligns with the current SDK behavior (`src/factpy_kernel/sdk/docs/00_user_guide.en.md`).  
Goal: cover as much current syntax and common usage as possible for quick onboarding and regression checks.

## 0. Imports

- Uses the public API from `factpy_kernel.sdk`.
- Demonstrates `SDKStore`, `Query`, `Rule`, `Derivation`, `ingest`, `registry`, and related paths.

In [1]:
from __future__ import annotations

import os
import shutil
import tempfile
import warnings
from pathlib import Path
from uuid import uuid4
from pprint import pprint

from factpy_kernel.sdk import (
    SDKStore,
    SDKRegistry,
    Entity,
    Identity,
    Field,
    Rule,
    RuleRef,
    Query,
    Derivation,
    Pred,
    Not,
    Body,
    vars,
    schema_preflight_from_classes,
    SDKStoreError,
    SDKSchemaError,
    CardinalityError,
    EntityNotFoundError,
    EditorClosedError,
)


## 1. Schema Definitions

This example schema covers:
- `Identity(primary_key/default_factory)`
- `Field(cardinality="single"|"multi")`
- entity reference fields (`entity_ref`)

In [2]:
class Country(Entity):
    iso_code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")


class Language(Entity):
    code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")


class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity()

    name: str = Field(cardinality="single")
    aliases: str = Field(cardinality="multi")
    age: int = Field(cardinality="single")
    country: Country = Field(cardinality="single")
    tag: str = Field(cardinality="multi")


class LivesIn(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    user: User = Field(cardinality="single")
    country: Country = Field(cardinality="single")
    since: int = Field(cardinality="single")


class HasLanguage(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    country: Country = Field(cardinality="single")
    language: Language = Field(cardinality="single")


class Speaks(Entity):
    uid: str = Identity(primary_key=True, default_factory="uuid4")
    user: User = Field(cardinality="single")
    language: Language = Field(cardinality="single")


## 2. Schema Preflight

`schema_preflight_from_classes(...)` can run schema health checks early in CI or at startup.

In [3]:
preflight = schema_preflight_from_classes([Country, Language, User, LivesIn, HasLanguage, Speaks])
print("preflight ok:", preflight["ok"])
pprint(preflight.get("summary", {}))
pprint(preflight.get("warnings", []))

preflight ok: True
{'entity_count': 6,
 'pred_ids': ['Country:exists',
              'HasLanguage:exists',
              'Language:exists',
              'LivesIn:exists',
              'Speaks:exists',
              'User:exists',
              'country:iso_code',
              'country:name',
              'has_language:country',
              'has_language:language',
              'has_language:uid',
              'language:code',
              'language:name',
              'lives_in:country',
              'lives_in:since',
              'lives_in:uid',
              'lives_in:user',
              'speaks:language',
              'speaks:uid',
              'speaks:user',
              'user:age',
              'user:aliases',
              'user:country',
              'user:locale',
              'user:name',
              'user:tag',
              'user:user_id'],
 'predicate_count': 27}
[]


## 3. Store Initialization Patterns

- in-memory store
- file-backed ledger
- `default_row_format` (Rule path only)

In [4]:
classes = [Country, Language, User, LivesIn, HasLanguage, Speaks]

# 1) In-memory
sdk = SDKStore.from_schema_classes(classes, default_row_format="dict")

# 2) File-backed ledger
ledger_path = Path(tempfile.gettempdir()) / f"factpy_example_ledger_{uuid4().hex}.db"
for suffix in ("", "-wal", "-shm"):
    p = Path(str(ledger_path) + suffix)
    if p.exists():
        p.unlink()

sdk_file = SDKStore.from_schema_classes(classes, ledger_path=str(ledger_path))
sdk_file_reopen = SDKStore.from_schema_classes(classes, ledger_path=str(ledger_path))

print("in-memory sdk:", type(sdk).__name__)
print("file ledger path:", ledger_path)

in-memory sdk: SDKStore
file ledger path: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/factpy_example_ledger_5388629526184f6fbbe8f2e3caa68ce8.db


### 3.1 `FACTPY_ROW_FORMAT` (read at SDK init time)

Only affects the Rule path; Query has its own `row_format` semantics.

In [5]:
os.environ["FACTPY_ROW_FORMAT"] = "tuple"
sdk_env = SDKStore.from_schema_classes(classes)

with vars("u") as (u,):
    trivial_rule = Rule(
        id="q.all_users",
        version="1.0.0",
        select=[u],
        where=[User(u)],
        expose=True,
    )

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", DeprecationWarning)
    _ = sdk_env.run(trivial_rule)

print("tuple deprecation warning count:", len(caught))

# cleanup env for later cells
os.environ.pop("FACTPY_ROW_FORMAT", None)

tuple deprecation warning count: 1


'tuple'

## 4. Batch Writes: `preview`, partial identity + `bind`, explicit `commit`

Note: `with sdk.batch() as tx:` does not auto-commit on `__exit__`.

In [6]:
tx = sdk.batch(meta={"trace_id": "seed-001", "source": "demo"})

# countries
de = tx.entity(Country, iso_code="DE")
de.name.set("Germany")

cn = tx.entity(Country, iso_code="CN")
cn.name.set("China")

# languages
de_lang = tx.entity(Language, code="de")
de_lang.name.set("German")

zh_lang = tx.entity(Language, code="zh")
zh_lang.name.set("Chinese")

en_lang = tx.entity(Language, code="en")
en_lang.name.set("English")

# user with full identity
u1 = tx.entity(User, user_id="u-001", locale="zh")
u1.name.set("Alice")
u1.aliases.add("Alicia")
u1.age.set(30)
u1.country.set(de)
u1.tag.add("vip")

# user with partial identity, then bind
u2 = tx.entity(User, user_id="u-002")
u2 = u2.bind(locale="en")
u2.name.set("Bob")
u2.aliases.add("Bobby")
u2.age.set(28)
u2.country.set(cn)
u2.tag.add("staff")

# relationship records
li1 = tx.entity(LivesIn)
li1.user.set(u1)
li1.country.set(de)
li1.since.set(2019)

li2 = tx.entity(LivesIn)
li2.user.set(u2)
li2.country.set(cn)
li2.since.set(2021)

hl1 = tx.entity(HasLanguage)
hl1.country.set(de)
hl1.language.set(de_lang)

hl2 = tx.entity(HasLanguage)
hl2.country.set(cn)
hl2.language.set(zh_lang)

plan = tx.preview()
print("planned ops:", len(plan.ops))

seed_result = tx.commit()
print("written assertion ids:", len(seed_result.apply_result.assertion_ids))

planned ops: 47
written assertion ids: 49


In [7]:
# Context manager does NOT auto-commit.
with sdk.batch() as tx_no_commit:
    ghost = tx_no_commit.entity(User, user_id="u-no-commit", locale="zh")
    ghost.name.set("WillNotPersist")

print("no-commit user exists?", sdk.get(User, user_id="u-no-commit", locale="zh") is not None)

no-commit user exists? False


### 4.1 Identity immutability and incomplete identity errors

In [8]:
with sdk.batch() as tx_err:
    incomplete = tx_err.entity(User, user_id="u-003")
    try:
        incomplete.name.set("Charlie")
    except SDKStoreError as exc:
        print("incomplete identity write blocked:", exc)

    incomplete = incomplete.bind(locale="fr")
    incomplete.name.set("Charlie")
    tx_err.commit()

with sdk.edit(User, user_id="u-001", locale="zh") as editor:
    try:
        editor.locale.set("en")
    except SDKStoreError as exc:
        print("identity immutable in editor:", exc)

incomplete identity write blocked: User#1.name: identity is incomplete for User; missing: ['locale']. Bind missing identity via handle.bind(...).
identity immutable in editor: identity field 'locale' is immutable in editor; open a new editor with different identity instead


## 5. Low-level writes: `ref / set / add / retract`

In [9]:
u1_ref = sdk.ref(User, user_id="u-001", locale="zh")

asrt_alias = sdk.add(
    User.aliases,
    u1_ref,
    "Alice Cooper",
    meta={"source": "low-level", "trace_id": "ll-1"},
)

asrt_age = sdk.set(
    User.age,
    u1_ref,
    31,
    meta={
        "source": "low-level",
        "trace_id": "ll-2",
        "version": "v2",
        "valid_from": "2024-01-01T00:00:00+00:00",
    },
)

revoker = sdk.retract(asrt_alias, meta={"source": "low-level", "trace_id": "ll-3"})

print("alias asrt:", asrt_alias)
print("age asrt:", asrt_age)
print("revoker asrt:", revoker)

alias asrt: 964148678d0e435b8cd5bbbd05b08392
age asrt: d5af793247a345d285f92916d0a9492b
revoker asrt: 99152cbc220245ba8e10b26432f98e4f


## 6. Read APIs: `get`, `find`, `EntitySnapshot`

In [10]:
snap = sdk.get(User, user_id="u-001", locale="zh")
print("ref:", snap.ref)
print("name:", snap.name)
print("aliases:", snap.aliases)
print("age:", snap.age)
print("country(ref):", snap.country)

rows_vip = sdk.find(User, tag="vip", limit=20)
print("vip refs:", [row.ref for row in rows_vip])

rows_identity = sdk.find(User, user_id="u-001", locale="zh")
print("identity-filter result count:", len(rows_identity))

ref: idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba
name: Alice
aliases: ('Alicia',)
age: 31
country(ref): idref_v1:Country:6yld57v2sjmk2up44c6xufn6gul22a5gclpagrqsx3rx53fk6qtq
vip refs: ['idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba']
identity-filter result count: 1


In [11]:
snap = sdk.get(User, user_id="u-001", locale="zh")

print("age.active count:", len(snap.assertions.age.active))
print("age.history count:", len(snap.assertions.age.history))
print("age.version('v2'):", [r.value for r in snap.assertions.age.version("v2")])
print("age.at('2024-06-01T00:00:00+00:00'):", [r.value for r in snap.assertions.age.at("2024-06-01T00:00:00+00:00")])

try:
    _ = snap.assertions.locale
except AttributeError as exc:
    print("identity field not in assertions namespace:", exc)

age.active count: 2
age.history count: 2
age.version('v2'): [31]
age.at('2024-06-01T00:00:00+00:00'): [31]
identity field not in assertions namespace: locale


## 7. Edit API: `sdk.edit(...)`

- auto-commit on normal exit
- rollback on exceptional path
- cardinality violations raise `CardinalityError` at the editor layer

In [12]:
with sdk.edit(User, user_id="u-001", locale="zh") as editor:
    editor.aliases.add("Alice Z")
    editor.age.set(32)

try:
    with sdk.edit(User, user_id="u-001", locale="zh") as editor:
        editor.age.add(999)
except CardinalityError as exc:
    print("cardinality blocked:", exc)

ed = sdk.edit(User, user_id="u-001", locale="zh")
ed.rollback()
try:
    ed.age.set(1)
except EditorClosedError as exc:
    print("closed editor blocked:", exc)

cardinality blocked: field 'age' has cardinality=single; .add is only valid for multi
closed editor blocked: editor is closed


## 8. Ingest API: external normalized items

In [13]:
snap_before_ingest = sdk.get(User, user_id="u-001", locale="zh")
old_alias_asrt_id = (
    snap_before_ingest.assertions.aliases.active[0].asrt_id
    if snap_before_ingest.assertions.aliases.active
    else None
)

items = [
    {"kind": "add", "field": User.tag, "e_ref": u1_ref, "value": "priority"},
    {
        "kind": "set",
        "field": User.age,
        "e_ref": u1_ref,
        "value": 33,
        "meta": {"version": "ingest-v3", "valid_from": "2025-01-01T00:00:00+00:00"},
    },
]
if old_alias_asrt_id is not None:
    items.append({"kind": "retract", "asrt_id": old_alias_asrt_id})

ingest_result = sdk.ingest(items, meta={"source": "ingest-demo", "trace_id": "ing-001"})
print("written:", len(ingest_result.written_assertion_ids))
print("duplicates:", ingest_result.duplicate_count, "skipped:", ingest_result.skipped_count)
pprint(ingest_result.warnings)
pprint(ingest_result.diagnostics)

written: 3
duplicates: 0 skipped: 0
[]
[]


## 9. Rule DSL (`sdk.run(rule, row_format=...)`)

In [14]:
with vars("u","vip","n") as (u,vip,n):
    vip_rule = Rule(
        id="q.vip_not_blocked",
        version="1.0.0",
        select=[vip],
        where=[
            Pred("user:tag", u, vip),
            Not([Pred("user:tag", u, "blocked")]),
        ],
        expose=True,
    )

    tag = Rule(
        id="name",
        version="1.0.0",
        select=[n],
        where=[
            User(u),
            u.name == n,
        ],
    )

    vip_ref_rule = Rule(
        id="q.vip_ref",
        version="1.0.0",
        select=[u],
        where=[RuleRef(vip_rule)(u)],
    )

vip_rows_dict = sdk.run(vip_rule, row_format="dict")
print("vip rows(dict):", vip_rows_dict)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", DeprecationWarning)
    vip_rows_tuple = sdk.run(vip_rule, row_format="tuple")

print("vip rows(tuple):", vip_rows_tuple)
print("tuple warnings:", len(caught))

print("name rows:", sdk.run(tag, row_format="dict"))
print("RuleRef rows:", sdk.run(vip_ref_rule, row_format="dict"))

vip rows(dict): [{'vip': 'priority'}, {'vip': 'staff'}, {'vip': 'vip'}]
vip rows(tuple): [('priority',), ('staff',), ('vip',)]
tuple warnings: 1
name rows: [{'n': 'Alice'}, {'n': 'Bob'}, {'n': 'Charlie'}]
RuleRef rows: [{'u': 'priority'}, {'u': 'staff'}, {'u': 'vip'}]


## 10. Query DSL (`dict` + `instance` modes)

Current behavior:
- `row_format="dict"` (default) returns dict rows
- `row_format="instance"` returns instance rows when the head is a single `Entity(var)`

In [15]:
from pprint import pp
with vars("u", "loc", "nm") as (u, loc, nm):
    q_dict = Query(
        head=[User(u), User.name(locale=loc, name=nm)],
        where=[User(u), u.locale == loc, u.name == nm],
        on_missing="error",
        on_type_mismatch="error",
    )

dict_rows = sdk.run(q_dict)
print("query dict rows:")
pp(dict_rows)

with vars("u", "loc") as (u, loc):
    q_instance = Query(
        head=User(u),
        where=[User(u), u.locale == loc, loc == "zh"],
    )

instance_rows = sdk.run(q_instance, row_format="instance")
print("query instance refs:")
pp([row.ref if row is not None else None for row in instance_rows])

with vars("u", "loc", "nm") as (u, loc, nm):
    bad_instance = Query(
        head=[User(u), User.name(locale=loc, name=nm)],
        where=[User(u), u.locale == loc, u.name == nm],
    )

try:
    sdk.run(bad_instance, row_format="instance")
except SDKStoreError as exc:
    print("bad instance head blocked:", exc.code, exc)

query dict rows:
[{'u': EntitySnapshot(entity_type='User', ref='idref_v1:User:42uubjiy5t3miixsx5pm5dmpc5ix37lluxopopkiyv4e37i3rk7q', age=28, aliases=('Bobby',), country='idref_v1:Cou...vwlzkfquddcfa', locale='en', name='Bob', tag=('staff',), user_id='u-002'),
  'nm': 'Bob'},
 {'u': EntitySnapshot(entity_type='User', ref='idref_v1:User:7hw2wb6dddwebueiaowvubvvt36bea3tcmlpop7fbzt74nzdyfqa', age=None, aliases=(), country=None, locale='fr', name='Charlie', tag=(), user_id='u-003'),
  'nm': 'Charlie'},
 {'u': EntitySnapshot(entity_type='User', ref='idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba', age=33, aliases=('Alice Z',), country='idref_v1:Cou...sx3rx53fk6qtq', locale='zh', name='Alice', tag=('priority', 'vip'), user_id='u-001'),
  'nm': 'Alice'}]
query instance refs:
['idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba']
bad instance head blocked: QUERY_INVALID_ROW_FORMAT Query return_mode='instance' requires exactly one Entity(var) item in head


## 11. Derivation DSL: `evaluate / accept / accept_many`(v3: `native/souffle/problog`)


In [16]:
with vars("u", "loc", "nm") as (u, loc, nm):
    alias_drv = Derivation(
        id="drv.derived_tag",
        version="1.0.0",
        where=[User(u), u.locale == loc, u.name == nm],
        head=User.tag(locale=loc, tag=nm),
    )

cands = sdk.evaluate(alias_drv, mode="native")
print("candidates:", len(cands))
print("candidate kinds:", {c.candidate_kind for c in cands})

if cands:
    dry = sdk.accept(cands[0], approved_by="demo", note="apply", dry_run=True)
    print("dry run accept:", dry)

    applied = sdk.accept(cands[0], approved_by="demo", note="apply")
    print("applied accept:", applied)

    duplicate = sdk.accept(cands[0], approved_by="demo", note="apply")
    print("duplicate accept (idempotent):", duplicate)


candidates: 3
candidate kinds: {'fact'}
dry run accept: AcceptResult(run_id='368789ebf351440fbaa17b24a7325bfd', accepted_count=1, skipped_count=0, written_assertions=[{'asrt_id': '<dry_run>', 'pred_id': 'user:tag', 'key_tuple_digest': 'sha256:87075cae6df5cf476dead778fe00835bf0fa8e2723e226cca8821a85e5049633'}], skipped_reason_counts={}, diagnostics=[], diagnostics_contract_version=1, entity_ref=None, candidate_id='cand_v2:f71f153e07a2d965a0887dde930f87f869b6e952ecaf654e213336a9dae08867', candidate_key='candk_v2:1357e45b5654a4a9bc1fdf057aebfdfb9e7544a655a58dd6d2b10e1fbb92f044')
applied accept: AcceptResult(run_id='368789ebf351440fbaa17b24a7325bfd', accepted_count=1, skipped_count=0, written_assertions=[{'asrt_id': '5b456903cbe5405382ec9c93978d8a67', 'pred_id': 'user:tag', 'key_tuple_digest': 'sha256:87075cae6df5cf476dead778fe00835bf0fa8e2723e226cca8821a85e5049633'}], skipped_reason_counts={}, diagnostics=[], diagnostics_contract_version=1, entity_ref=None, candidate_id='cand_v2:f71f153e0

In [17]:
with vars("u", "loc", "nm", "tg") as (u, loc, nm, tg):
    multi_head_drv = Derivation(
        id="drv.multi",
        version="1.0.0",
        where=[User(u), u.locale == loc, u.name == nm, u.tag == tg],
        head=[
            User.name(locale=loc, name=nm),
            User.tag(locale=loc, tag=tg),
        ],
    )

multi_cands = sdk.evaluate(multi_head_drv, mode="native")
print("multi-head candidates:", len(multi_cands))
print("multi-head run_ids:", {c.run_id for c in multi_cands})

if multi_cands:
    many = sdk.accept_many(multi_cands[:2], mode="atomic")
    print("accept_many:", many)

try:
    sdk.evaluate(alias_drv, temporal_view="active")
except SDKStoreError as exc:
    print("temporal_view rejected in evaluate:", exc)

try:
    sdk.run(alias_drv)
except SDKStoreError as exc:
    print("run(derivation) rejected:", exc.code, exc)


multi-head candidates: 6
multi-head run_ids: {'drv.multi:08ea9d55'}
accept_many: [{'candidate_id': 'cand_v2:a3c89add30029bd381c8dc71e51fb01ec7bfabe0f4bda8a150b20a10f107b083', 'candidate_key': 'candk_v2:3a9383b22ec683e8ac84463089053d8ff9d922f404727d8f30526fd091acfcae', 'state': 'ACCEPTED', 'entity_ref': None, 'error': None}, {'candidate_id': 'cand_v2:ad18ffd1bc60059b4605d9f249bee6bc169b10067a6ee086797c05821d1ea1be', 'candidate_key': 'candk_v2:80edfe03e534824db025069cb4f288a320253856936f63c5aac2be4d2ee16bf2', 'state': 'ACCEPTED', 'entity_ref': None, 'error': None}]
temporal_view rejected in evaluate: temporal_view is removed from evaluate(); use active/history views on read APIs
run(derivation) rejected: QUERY_INVALID_ROW_FORMAT Derivation is not supported by run(); use sdk.evaluate() instead


### 11.1 v3 Syntax Addendum: `Body`, `view`, and optional `problog`

- `Body([...], confidence=...)` models explicit confidence on OR branches (`Query` does not support `Body.confidence`).
- `sdk.run(rule, view=..., return_display_meta=True)` returns display-level aggregated confidence metadata.
- `mode="problog"` is optional; if the CLI is unavailable, this demo falls back gracefully.


In [18]:
from factpy_kernel.core.store.types import ViewSpec

# Write two confidence values for the same fact to demonstrate view aggregation
sdk.add(User.tag, u1_ref, "vip", meta={"source": "profile", "trace_id": "conf-1", "confidence": 0.82})
sdk.add(User.tag, u1_ref, "vip", meta={"source": "model", "trace_id": "conf-2", "confidence": 0.64})

# Body + confidence (native ignores body_confidences, but syntax and plumbing are valid)
with vars("u", "loc", "tg") as (u, loc, tg):
    body_drv = Derivation(
        id="drv.body_demo",
        version="1.0.0",
        where=[
            Body([User(u), u.locale == loc, Pred("user:tag", u, tg)], confidence=0.9),
            Body([User(u), Pred("user:tag", u, tg)], confidence=0.6),
        ],
        head=User.tag(locale=loc, tag=tg),
    )

body_cands = sdk.evaluate(body_drv, mode="native")
print("body(native) candidates:", len(body_cands))

# run(view=...) + return_display_meta
sdk.views.create("mean_conf", ViewSpec(confidence_strategy="mean"))
rows, display_meta = sdk.run(vip_rule, view="mean_conf", return_display_meta=True, row_format="dict")
print("run(view) row count:", len(rows))
print("run(view) display meta sample:", display_meta[0] if display_meta else None)

# Optional: ProbLog (should raise a controlled error if CLI is unavailable)
try:
    import factpy_kernel.adapters.problog  # noqa: F401
    prob_cands = sdk.evaluate(body_drv, mode="problog")
    print("problog candidates:", len(prob_cands), "confidence sample:", prob_cands[0].confidence if prob_cands else None)
except Exception as exc:
    print("problog demo skipped:", type(exc).__name__, exc)



body(native) candidates: 4
run(view) row count: 4
run(view) display meta sample: {'confidence': None, 'confidence_strategy': 'mean', 'source_breakdown': []}
problog demo skipped: ProbLogEngineError ProbLog CLI failed with exit code 1


## 12. Provenance Validation (`sdk.validate_provenance`)

In [19]:
report_ok = sdk.validate_provenance(
    {
        "derived_rule_id": "drv.derived_tag",
        "derived_rule_version": "1.0.0",
        "run_id": "run-demo-001",
        "support_kind": "tuple",
        "support_digest": "sha256:" + "a" * 64,
    },
    standard="derivation_v1",
)

report_bad = sdk.validate_provenance(
    {
        "derived_rule_id": "",
        "derived_rule_version": "1.0.0",
        "run_id": "",
        "support_kind": "tuple",
        "support_digest": "not-a-sha256-token",
    },
    standard="derivation_v1",
)

print("report_ok:", report_ok.ok)
print("report_bad:", report_bad.ok)
pp(report_bad.errors)

report_ok: True
report_bad: False
[{'code': 'provenance_required_field_missing_or_invalid',
  'severity': 'error',
  'path': '$.provenance.derived_rule_id',
  'message': 'derived_rule_id must be non-empty string',
  'data': {'key': 'derived_rule_id'}},
 {'code': 'provenance_required_field_missing_or_invalid',
  'severity': 'error',
  'path': '$.provenance.run_id',
  'message': 'run_id must be non-empty string',
  'data': {'key': 'run_id'}},
 {'code': 'provenance_required_digest_missing_or_invalid',
  'severity': 'error',
  'path': '$.provenance.support_digest',
  'message': "support_digest must be 'sha256:<hex>'",
  'data': {'key': 'support_digest'}}]


## 13. Audit APIs: `explain_fact` and `conflicts`

In [20]:
def pred_id_for(sdk_obj: SDKStore, owner_type: str, field_name: str) -> str:
    for pred in sdk_obj.schema_ir.get("predicates", []):
        if not isinstance(pred, dict):
            continue
        if pred.get("owner_type") != owner_type:
            continue
        if pred.get("py_field_name") != field_name:
            continue
        pred_id = pred.get("pred_id")
        if isinstance(pred_id, str) and pred_id:
            return pred_id
    raise RuntimeError(f"pred_id not found for {owner_type}.{field_name}")

user_tag_pred = pred_id_for(sdk, "User", "tag")
explain = sdk.explain_fact(user_tag_pred, u1_ref, "vip")
conf = sdk.conflicts(user_tag_pred, u1_ref)

print("pred_id:", user_tag_pred)
pprint(explain)
pprint(conf)

pred_id: user:tag
{'active_claims': [{'args': ('idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba',
                             'vip'),
                    'asrt_id': '9b9277758e66434580b7a83f5e51edbd',
                    'meta': {'ingested_at': 1772637557443535000,
                             'source': 'demo'}},
                   {'args': ('idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba',
                             'vip'),
                    'asrt_id': '209e555b140543169bee2288ec31d529',
                    'meta': {'ingested_at': 1772637557560345000,
                             'source': 'profile'}},
                   {'args': ('idref_v1:User:mpdat3awltdenidj64aocoihw44q7n4m5tzu3mmnld5wimwtc7ba',
                             'vip'),
                    'asrt_id': 'b5ab00d53e634af7a8f5c30567f01e58',
                    'meta': {'ingested_at': 1772637557561716000,
                             'source': 'model'}}],
 'chosen_asrt_id': '209e55

## 14. Registry Example (`SDKRegistry`)

- `apply_schema_classes`
- `register_rule`
- `register_derivation`
- read/list APIs

In [21]:
registry_root = Path(tempfile.gettempdir()) / "factpy_registry_demo"
if registry_root.exists():
    shutil.rmtree(registry_root)

registry = SDKRegistry(root_dir=registry_root)
apply_out = registry.apply_schema_classes(classes)
print("apply_schema_classes ok:", apply_out.get("ok"))
print("schema entry:", registry.get_schema_entry())

_ = registry.register_rule(vip_rule, schema_ir=sdk.schema_ir)
_ = registry.register_derivation(alias_drv, schema_ir=sdk.schema_ir)

print("rule ids:", registry.list_rule_ids())
print("derivation ids:", registry.list_derivation_ids())
print("latest q.vip exists:", registry.get_latest_rule_spec("q.vip") is not None)
print("latest drv.derived_tag exists:", registry.get_latest_derivation_spec("drv.derived_tag") is not None)

apply_schema_classes ok: True
schema entry: {'bytes_digest': 'sha256:854b8c6d384cba5574b7197b26edc69409e7c680bee191f805f17c3c3d1aa44d', 'path': 'schema/schema_ir.json', 'schema_digest': 'sha256:854b8c6d384cba5574b7197b26edc69409e7c680bee191f805f17c3c3d1aa44d'}
rule ids: ['q.vip_not_blocked']
derivation ids: ['drv.derived_tag']
latest q.vip exists: False
latest drv.derived_tag exists: True


## 15. Common boundary checks (expected errors)

These examples highlight current SDK boundaries, not recommended style.

In [22]:
try:
    sdk.run("select * from ...")
except SDKStoreError as exc:
    print("string rule DSL rejected:", exc)

try:
    sdk.evaluate("derive ...")
except SDKStoreError as exc:
    print("string derivation DSL rejected:", exc)

try:
    _ = sdk.get(User, user_id="u-001")  # missing locale identity
except SDKSchemaError as exc:
    print("get missing identity rejected:", exc)

try:
    sdk.edit(User, user_id="u-not-found", locale="zh")
except EntityNotFoundError as exc:
    print("edit missing entity:", exc)


string rule DSL rejected: string rule DSL is not supported in SDK v1; use Rule object, RuleSpec, or structured rule dict
string derivation DSL rejected: string derivation DSL is not supported in SDK v1; use Derivation object or structured derivation dict
get missing identity rejected: missing identity fields for get(User): ['locale']
edit missing entity: entity not found: User with identity {'user_id': 'u-not-found', 'locale': 'zh'}


## 16. Notes

- `single` fields are a read-side single-value view; old assertions are not auto-cleaned.  
- `snapshot.assertions` covers `Field` attributes only, not `Identity` attributes.  
- Query `instance` mode is best for entity-list queries; use dict rows for complex projections.